<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">湖表与内部表关联</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">候选实验：需要讲师预置真实 Iceberg 环境，当前尚未实测。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course4_querying_external_data.md) · [课程入口](../README.md)


## 环境要求与状态

这是需要外部环境的候选 Lab，首版不提供 Iceberg 服务部署。讲师需预置包含 datasets/orders.json 全部字段的 Iceberg orders 表，并通过 Doris Catalog 可查询。设置 DW_ICEBERG_ORDERS=catalog.database.table；没有该环境时应明确记录未执行，不用内部表冒充湖表。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import os
from dw_course.runtime import identifier
source_parts = os.environ["DW_ICEBERG_ORDERS"].split(".")
if len(source_parts) != 3:
    raise ValueError("Expected catalog.database.table")
source = ".".join(identifier(part) for part in source_parts)
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 直查湖表

同一份 10 笔订单应有 1400.00 金额。这里没有导入数据；Catalog 提供外部表元数据。


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"1400.00")])
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1001");


## 2. 与内部客户表关联

每个订单客户对应一个客户行；关联后数量不能扩大。仅重建 d04_customers 与 d04_orders。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d04_customers")
lab.execute('CREATE TABLE d04_customers (customer_id BIGINT, region VARCHAR(16)) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("d04_customers", ["customer_id","region"], [(r["customer_id"],r["region"]) for r in fixture("orders.json")])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN d04_customers c ON o.customer_id=c.customer_id"), [(10,"1400.00")])
lab.execute("DROP TABLE IF EXISTS d04_orders")
ddl = order_ddl("d04_orders")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO d04_orders ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d04_orders"), [(10,"1400.00")])
lab.close()


## 完成与边界

记录 Catalog 类型和外部服务版本、原表标识、结果与计划。独立 Parquet 的 S3 TVF 实验仍待补；读取成功不代表外部写入、Schema 演进或性能 SLA 已验证。
